In [0]:
-- SCD Type 2 load: expire-and-insert for CHANGED customers, plain insert for NEW customers.
-- A Delta MERGE can apply only one action per source row, so CHANGED rows are staged twice:
--   * one copy carries merge_key = customer_id  -> matches the current target row and closes it
--   * one copy carries merge_key = NULL          -> never matches (NULL comparison) and inserts the new version
-- NEW customers appear only in the insert copy, since no current target row exists to close.
-- 05_load
-- Loads dim_customer and dim_product via SCD Type 2 MERGE, loads dim_channel
-- as a stable lookup, then loads dim_date (no SCD) and fact_orders.
-- fact_orders resolves customer_sk and product_sk with point-in-time joins,
-- and resolves channel_sk with a direct channel_name lookup.
-- Source: northriver_staging (xform_* tables) | Target: northriver_warehouse
-- Run after: 04_transform
WITH changes AS (
  SELECT *
  FROM northriver_staging.xform_dim_customer
  WHERE change_type IN ('NEW', 'CHANGED')
)
MERGE INTO northriver_warehouse.dim_customer AS t
USING (
  -- Copy 1: CHANGED customers that must expire their current row
  SELECT c.*, c.customer_id AS merge_key
  FROM changes c
  WHERE c.change_type = 'CHANGED'

  UNION ALL

  -- Copy 2: rows to insert (new customers and new versions of changed customers)
  SELECT c.*, NULL AS merge_key
  FROM changes c
) AS s
ON t.customer_id = s.merge_key
AND t.is_current = TRUE

-- Close the old (current) row for changed customers
WHEN MATCHED THEN
  UPDATE SET
    t.effective_to = current_date() - INTERVAL 1 DAY,
    t.is_current   = FALSE

-- Insert the new row (new customer or new version of a changed customer)
WHEN NOT MATCHED THEN
  INSERT (
    customer_id,
    first_name,
    last_name,
    email,
    region,
    segment,
    effective_from,
    effective_to,
    is_current
  )
  VALUES (
    s.customer_id,
    s.first_name,
    s.last_name,
    s.email,
    s.region,
    s.segment,
    COALESCE(CAST(s.source_updated_at AS DATE), current_date()),
    NULL,
    TRUE
  );
-- SCD Type 2 load: expire-and-insert for CHANGED products, plain insert for NEW products.
WITH changes AS (
  SELECT *
  FROM northriver_staging.xform_dim_product
  WHERE change_type IN ('NEW', 'CHANGED')
)
MERGE INTO northriver_warehouse.dim_product AS t
USING (
  -- Copy 1: CHANGED products that must expire their current row
  SELECT c.*, c.product_id AS merge_key
  FROM changes c
  WHERE c.change_type = 'CHANGED'

  UNION ALL

  -- Copy 2: rows to insert (new products and new versions of changed products)
  SELECT c.*, NULL AS merge_key
  FROM changes c
) AS s
ON t.product_id = s.merge_key
AND t.is_current = TRUE

-- Close the old (current) row for changed products
WHEN MATCHED THEN
  UPDATE SET
    t.effective_to = current_date() - INTERVAL 1 DAY,
    t.is_current   = FALSE

-- Insert the new row (new product or new version of a changed product)
WHEN NOT MATCHED THEN
  INSERT (
    product_id,
    product_name,
    brand,
    category,
    subcategory,
    unit_cost,
    effective_from,
    effective_to,
    is_current
  )
  VALUES (
    s.product_id,
    s.product_name,
    s.brand,
    s.category,
    s.subcategory,
    s.unit_cost,
    current_date(),
    NULL,
    TRUE
  );
-- SCD Type 2 load: plain insert for NEW channels (no CHANGED case)
WITH changes AS (
  SELECT *
  FROM northriver_staging.xform_dim_channel
  WHERE change_type = 'NEW'
)
MERGE INTO northriver_warehouse.dim_channel AS t
USING changes AS s
ON t.channel_name = s.channel_name

WHEN NOT MATCHED THEN
  INSERT (
    channel_name
  )
  VALUES (
    s.channel_name
  );

  SELECT COUNT(*) FROM northriver_warehouse.dim_customer;

-- dim_date load: the calendar is deterministic (no identity, no SCD), so an
-- idempotent insert keyed on date_sk — safe to re-run, like the channel load.
WITH changes AS (
  SELECT *
  FROM northriver_staging.xform_dim_date
)
MERGE INTO northriver_warehouse.dim_date AS t
USING changes AS s
ON t.date_sk = s.date_sk

WHEN NOT MATCHED THEN
  INSERT (
    date_sk,
    full_date,
    year,
    quarter,
    month,
    day_of_week
  )
  VALUES (
    s.date_sk,
    s.full_date,
    s.year,
    s.quarter,
    s.month,
    s.day_of_week
  );

MERGE INTO northriver_warehouse.fact_orders AS t
USING (
  SELECT
    xf.order_id,
    xf.order_item_id,
    dc.customer_sk,
    dp.product_sk,
    dchn.channel_sk,
    dd.date_sk,
    xf.quantity,
    xf.unit_price,
    xf.quantity * xf.unit_price AS revenue
  FROM northriver_staging.xform_fact_orders xf
  JOIN northriver_warehouse.dim_customer dc
    ON xf.customer_id = dc.customer_id
    AND xf.order_date BETWEEN dc.effective_from AND COALESCE(dc.effective_to, DATE '9999-12-31')
    AND dc.is_current = TRUE
  JOIN northriver_warehouse.dim_product dp
    ON xf.product_id = dp.product_id
    AND xf.order_date BETWEEN dp.effective_from AND COALESCE(dp.effective_to, DATE '9999-12-31')
    AND dp.is_current = TRUE
  JOIN northriver_warehouse.dim_channel dchn
    ON xf.channel = dchn.channel_name
  JOIN northriver_warehouse.dim_date dd
    ON xf.order_date = dd.full_date
) AS s
ON t.order_item_id = s.order_item_id

WHEN NOT MATCHED THEN
  INSERT (
    order_id,
    order_item_id,
    customer_sk,
    product_sk,
    channel_sk,
    date_sk,
    quantity,
    unit_price,
    revenue
  )
  VALUES (
    s.order_id,
    s.order_item_id,
    s.customer_sk,
    s.product_sk,
    s.channel_sk,
    s.date_sk,
    s.quantity,
    s.unit_price,
    s.revenue
  );



  SELECT COUNT(*)     FROM northriver_warehouse.fact_orders;  -- 32
SELECT SUM(revenue) FROM northriver_warehouse.fact_orders;  -- should match Sprint 2 total